In [1]:
import os 
from dotenv import load_dotenv
from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient

from sentence_transformers import CrossEncoder
from openai import OpenAI

load_dotenv()


True

In [2]:
COLLECTION_NAME    = "Thesis Guide for USU Computer Science Students"              # Nama collection Qdrant
EMBEDDING_MODEL    = "paraphrase-multilingual-MiniLM-L12-v2"

# Retrieval
TOP_K              = 25                      # Jumlah vektor yang di-retrieve
TOP_N_RERANK       = 10                       # Jumlah hasil setelah reranking

# NVIDIA
NVIDIA_MODEL = "meta/llama-3.1-8b-instruct"
NVIDIA_API_KEY = os.getenv("NVIDIA_API_KEY")
NVIDIA_BASE_URL = os.getenv("NVIDIA_BASE_URL")                                    # (~375 token; 5 chunk × 1500 ≈ 1875 token context)
MAX_TOKENS_OUTPUT   = 2048                     # Maks token jawaban LLM

HF_TOKEN = os.getenv("HF_TOKEN")

# Credentials dari .env
QDRANT_URL         = os.getenv("QDRANT_URL")
QDRANT_API_KEY     = os.getenv("QDRANT_API_KEY")

# Validasi
assert QDRANT_URL,     "❌ QDRANT_URL tidak ditemukan di .env!"
assert QDRANT_API_KEY, "❌ QDRANT_API_KEY tidak ditemukan di .env!"
assert NVIDIA_API_KEY,   "❌ NVIDIA_API_KEY tidak ditemukan di .env!"
assert NVIDIA_MODEL,   "❌ NVIDIA_MODEL tidak ditemukan di .env!"
assert NVIDIA_BASE_URL,   "❌ NVIDIA_BASE_URL tidak ditemukan di .env!"


print("✅ Konfigurasi berhasil dimuat!")
print(f"   🤖 NVIDIA_MODEL Model         : {NVIDIA_MODEL}")
print(f"   📤 Max tokens output  : {MAX_TOKENS_OUTPUT}")

✅ Konfigurasi berhasil dimuat!
   🤖 NVIDIA_MODEL Model         : meta/llama-3.1-8b-instruct
   📤 Max tokens output  : 2048


In [3]:
embed_model = SentenceTransformer(EMBEDDING_MODEL)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [4]:
# ─── Masukkan pertanyaan di sini ──────────────────────────────────────────────
user_query = "Apa langkah-langkah maju seminar proposal?"

if not user_query:
    raise ValueError("❌ Query tidak boleh kosong!")

print(f"\n📝 Query: {user_query}")


📝 Query: Apa langkah-langkah maju seminar proposal?


In [5]:
print("⏳ Melakukan embedding query ...")

query_embedding = embed_model.encode(
    user_query,
    normalize_embeddings=True
).tolist()

print(f"✅ Query berhasil di-embed! Dimensi: {len(query_embedding)}")

⏳ Melakukan embedding query ...
✅ Query berhasil di-embed! Dimensi: 384


In [6]:
print(f"⏳ Menghubungkan ke Qdrant di {QDRANT_URL} ...")
qdrant = QdrantClient(
    url=QDRANT_URL,
    api_key=QDRANT_API_KEY,
    timeout=60
)
print("✅ Koneksi berhasil!")

⏳ Menghubungkan ke Qdrant di https://ea295c4d-3efb-4676-9964-a20d09b4e162.us-east4-0.gcp.cloud.qdrant.io ...
✅ Koneksi berhasil!


In [7]:
print(f"⏳ Melakukan retrieval top-{TOP_K} dari Qdrant ...")

# qdrant-client v2.x: gunakan query_points (pengganti .search yang deprecated)
search_results = qdrant.query_points(
    collection_name=COLLECTION_NAME,
    query=query_embedding,
    limit=TOP_K,
    with_payload=True
).points

retrieved_chunks = [
    {
        "id"     : hit.id,
        "text"   : hit.payload["text"],
        "source" : hit.payload["source"],
        "score"  : hit.score
    }
    for hit in search_results
]

print(f"✅ Berhasil retrieve {len(retrieved_chunks)} chunks!")
print("\n📋 Top 5 hasil retrieval (sebelum reranking):")
print("-" * 60)
for i, chunk in enumerate(retrieved_chunks[:5], 1):
    print(f"[{i}] Score: {chunk['score']:.4f} | Sumber: {chunk['source']}")
    print(f"    {chunk['text'][:150]}...\n")

⏳ Melakukan retrieval top-25 dari Qdrant ...
✅ Berhasil retrieve 25 chunks!

📋 Top 5 hasil retrieval (sebelum reranking):
------------------------------------------------------------
[1] Score: 0.7223 | Sumber: Pedoman skripsi tambahan.md
    Telah menyelesaikan semua mata kuliah wajib. 5. Telah lulus mata kuliah Metodologi Penelitian. 6. Telah melakukan bimbingan proposal dengan kedua dose...

[2] Score: 0.6977 | Sumber: Pedoman skripsi tambahan.md
    ## 4. Tahap Pelaksanaan Seminar Proposal

* Penjadwalan: Pihak Prodi / Fakultas melakukan Penjadwalan Seminar Proposal. * Pelaksanaan: Prodi / Fakulta...

[3] Score: 0.6713 | Sumber: Prosedur Tugas Akhir Prodi S-1 Ilmu Komputer.md
    Penelitian

9. Exum (Judul, Latar Belakang, Rumusan Masalah, Metodologi, Referensi)

10. Tanda tangan mahasiswa

---

# Seminar Proposal

Untuk maju s...

[4] Score: 0.6502 | Sumber: Prosedur Tugas Akhir Prodi S-1 Ilmu Komputer.md
    # Prosedur Tugas Akhir Prodi S-1 Ilmu Komputer

*Diperuntukkan Bagi Maha

In [8]:
RERANKER_MODEL = "cross-encoder/ms-marco-MiniLM-L-12-v2"

# ─── Inisialisasi CrossEncoder Reranker ──────────────────────────────────────
print(f"⏳ Memuat CrossEncoder reranker: {RERANKER_MODEL} ...")
cross_encoder = CrossEncoder(
    model_name=RERANKER_MODEL,
    max_length=512
)
print("✅ CrossEncoder reranker siap!")

The CrossEncoder `model_name` argument was renamed and is now deprecated. Please use `model_name_or_path` instead.


⏳ Memuat CrossEncoder reranker: cross-encoder/ms-marco-MiniLM-L-12-v2 ...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

✅ CrossEncoder reranker siap!


In [9]:
# ─── Buat pasangan (query, passage) untuk scoring ────────────────────────────
pairs = [[user_query, chunk["text"]] for chunk in retrieved_chunks]

# ─── Lakukan reranking ────────────────────────────────────────────────────────
print(f"\n⏳ Melakukan reranking {len(pairs)} chunks ...")
rerank_scores = cross_encoder.predict(pairs, show_progress_bar=True)

# Gabungkan score ke retrieved_chunks, lalu sort descending
for chunk, score in zip(retrieved_chunks, rerank_scores):
    chunk["rerank_score"] = float(score)

top_reranked = sorted(
    retrieved_chunks,
    key=lambda x: x["rerank_score"],
    reverse=True
)[:TOP_N_RERANK]

print(f"\n✅ Reranking selesai! Menampilkan top-{TOP_N_RERANK}:")
print("-" * 60)
for idx, result in enumerate(top_reranked, 1):
    print(f"[{idx}] Rerank Score : {result['rerank_score']:.4f} | Sumber: {result['source']}")
    print(f"     Vector Score: {result['score']:.4f}")
    print(f"     {result['text'][:150]}...\n")


⏳ Melakukan reranking 25 chunks ...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]


✅ Reranking selesai! Menampilkan top-10:
------------------------------------------------------------
[1] Rerank Score : 2.6720 | Sumber: Prosedur Tugas Akhir Prodi S-1 Ilmu Komputer.md
     Vector Score: 0.6713
     Penelitian

9. Exum (Judul, Latar Belakang, Rumusan Masalah, Metodologi, Referensi)

10. Tanda tangan mahasiswa

---

# Seminar Proposal

Untuk maju s...

[2] Rerank Score : 1.8700 | Sumber: Pedoman skripsi tambahan.md
     Vector Score: 0.6977
     ## 4. Tahap Pelaksanaan Seminar Proposal

* Penjadwalan: Pihak Prodi / Fakultas melakukan Penjadwalan Seminar Proposal. * Pelaksanaan: Prodi / Fakulta...

[3] Rerank Score : 0.6183 | Sumber: Pedoman skripsi tambahan.md
     Vector Score: 0.5883
     Pada masa pandemi Covid 19, kegiatan seminar mahasiswa baik itu seminar proposal, seminar hasil dan sidang sarjana dilakukan secara daring dengan plat...

[4] Rerank Score : 0.0699 | Sumber: Pedoman skripsi tambahan.md
     Vector Score: 0.6456
     3. Mahasiswa wajib menyiapkan di

In [10]:
# ─── Gabungkan context dari top reranked chunks ───────────────────────────────
context_parts = []
for idx, result in enumerate(top_reranked, 1):
    source = result["source"]
    text   = result["text"]
    context_parts.append(f"[Source {idx}: {source}]\n{text}")

context = "\n\n".join(context_parts)

# ─── Prompt Template ─────────────────────────────────────────────────────────
PROMPT_TEMPLATE = """Use the following pieces of information to answer the user's question.
If you don't know the answer, just say that you don't know, don't try to make up an answer.

Context:
{context}

Query: {query}

Answer the question and provide additional helpful information,
based on the pieces of information, if applicable. Be succinct.
Responses should be properly formatted to be easily read."""

final_prompt = PROMPT_TEMPLATE.format(
    context=context,
    query=user_query
)

print("✅ Prompt berhasil dibuat!")
print(f"   Total karakter prompt: {len(final_prompt):,}")
print("\n" + "=" * 60)
print("PREVIEW PROMPT:")
print("=" * 60)
print(final_prompt[:800] + "\n...[dipotong untuk preview]" if len(final_prompt) > 800 else final_prompt)

✅ Prompt berhasil dibuat!
   Total karakter prompt: 18,850

PREVIEW PROMPT:
Use the following pieces of information to answer the user's question.
If you don't know the answer, just say that you don't know, don't try to make up an answer.

Context:
[Source 1: Prosedur Tugas Akhir Prodi S-1 Ilmu Komputer.md]
Penelitian

9. Exum (Judul, Latar Belakang, Rumusan Masalah, Metodologi, Referensi)

10. Tanda tangan mahasiswa

---

# Seminar Proposal

Untuk maju seminar proposal mahasiswa perlu menyelesaikan dua submission di bawah ini secara berurutan. ## 1. Persetujuan Seminar Proposal

Mahasiswa diharuskan mendapatkan persetujuan untuk mengikuti seminar Proposal oleh dosen pembimbing melalui modul ini. Mahasiswa wajib mengirimkan file submission berupa :

1. File Proposal dengan format : PROPOSAL_NIM.pdf

2. File Form Persetujuan Seminar Proposal yang sudah diisi dan di
...[dipotong untuk preview]


In [11]:
# ─── Inisialisasi NVIDIA NIM client ──────────────────────────────────────────
nim_client = OpenAI(
    api_key=NVIDIA_API_KEY,
    base_url=NVIDIA_BASE_URL  # contoh: "https://integrate.api.nvidia.com/v1"
)

print(f"⏳ Mengirim prompt ke NVIDIA NIM ({NVIDIA_MODEL}) ...")

chat_completion = nim_client.chat.completions.create(
    model=NVIDIA_MODEL,  # contoh: "meta/llama-3.1-8b-instruct"
    messages=[
        {
            "role"   : "system",
            "content": "You are a helpful assistant that answers questions based on provided context."
        },
        {
            "role"   : "user",
            "content": final_prompt
        }
    ],
    temperature=0.2,
    max_tokens=1024,
    top_p=0.9
)

llm_answer = chat_completion.choices[0].message.content
usage      = chat_completion.usage

print("✅ Respons diterima dari NVIDIA NIM!")
print(f"   Prompt tokens     : {usage.prompt_tokens:,}")
print(f"   Completion tokens : {usage.completion_tokens:,}")
print(f"   Total tokens      : {usage.total_tokens:,}")

from IPython.display import Markdown, display
print("=" * 70)
print("🔍 PERTANYAAN:")
print("=" * 70)
print(user_query)
print("\n" + "=" * 70)
print(f"🤖 JAWABAN ({NVIDIA_MODEL}):")
print("=" * 70)
display(Markdown(llm_answer))

⏳ Mengirim prompt ke NVIDIA NIM (meta/llama-3.1-8b-instruct) ...
✅ Respons diterima dari NVIDIA NIM!
   Prompt tokens     : 5,587
   Completion tokens : 194
   Total tokens      : 5,781
🔍 PERTANYAAN:
Apa langkah-langkah maju seminar proposal?

🤖 JAWABAN (meta/llama-3.1-8b-instruct):


Berikut adalah langkah-langkah maju seminar proposal:

1. Mahasiswa harus menyelesaikan dua submission di bawah ini secara berurutan:
 * Persetujuan Seminar Proposal
 * Workshop Seminar Proposal

Pada tahap Persetujuan Seminar Proposal, mahasiswa harus mengirimkan file submission berupa:
 * File Proposal dengan format: PROPOSAL_NIM.pdf
 * File Form Persetujuan Seminar Proposal yang sudah diisi dan ditandatangani oleh Dosen Pembimbing dengan format: DOPING_NIM.pdf beserta dengan lembar kendali bimbingan pra proposal yang telah ditandatangani oleh dosen pembimbing

Pada tahap Workshop Seminar Proposal, mahasiswa diwajibkan melakukan submission kembali dengan proposal yang sudah siap dengan format: Proposal_NIM.pdf

Setelah itu, mahasiswa dapat melanjutkan ke tahap Seminar Hasil.

In [12]:
def ask(query: str) -> str:
    """Fungsi lengkap: query → embed → retrieve → rerank → generate → return answer."""

    # Embed query
    q_embedding = embed_model.encode(query, normalize_embeddings=True).tolist()

    # Retrieve top-K dari Qdrant
    hits = qdrant.query_points(
        collection_name=COLLECTION_NAME,
        query=q_embedding,
        limit=TOP_K,
        with_payload=True
    ).points

    candidates = [
        {
            "id"     : h.id,
            "text"   : h.payload["text"],
            "source" : h.payload["source"],
            "score"  : h.score
        }
        for h in hits
    ]

    # CrossEncoder reranking
    pairs  = [[query, c["text"]] for c in candidates]
    scores = cross_encoder.predict(pairs)
    for c, s in zip(candidates, scores):
        c["rerank_score"] = float(s)

    reranked = sorted(candidates, key=lambda x: x["rerank_score"], reverse=True)[:TOP_N_RERANK]

    # Build context
    context = "\n\n".join(
        f"[Source {i+1}: {r['source']}]\n{r['text']}"
        for i, r in enumerate(reranked)
    )

    # Build prompt
    prompt = PROMPT_TEMPLATE.format(context=context, query=query)

    resp = nim_client.chat.completions.create(
        model=NVIDIA_MODEL,  # contoh: "meta/llama-3.1-8b-instruct"
        messages=[
            {
                "role"   : "system",
                "content": "You are a helpful assistant that answers questions based on provided context."
            },
            {
                "role"   : "user",
                "content": prompt
            }
        ],
        temperature=0.2,
        max_tokens=1024,
        top_p=0.9
    )
    return resp.choices[0].message.content
    
# ─── Loop interaktif ──────────────────────────────────────────────────────────
print("💬 Mode Interaktif RAG — ketik 'keluar' untuk berhenti\n")
while True:
    q = input("🔍 Pertanyaan: ").strip()
    if q.lower() in ("keluar", "exit", "quit", "q"):
        print("👋 Sesi selesai.")
        break
    if not q:
        continue
    print("\n⏳ Memproses ...\n")
    answer = ask(q)
    from IPython.display import Markdown, display
    
    print("=" * 70)
    print("🔍 PERTANYAAN:")
    print("=" * 70)
    print(q)
    
    print("\n" + "=" * 70)
    print(f"🤖 JAWABAN  ({NVIDIA_MODEL}):")
    print("=" * 70)
    
    display(Markdown(answer))

💬 Mode Interaktif RAG — ketik 'keluar' untuk berhenti


⏳ Memproses ...

🔍 PERTANYAAN:
apa langkah-langkah maju seminar proposal?

🤖 JAWABAN  (meta/llama-3.1-8b-instruct):


Berikut adalah langkah-langkah maju seminar proposal berdasarkan informasi yang tersedia:

1. **Persetujuan Seminar Proposal**: Mahasiswa harus mendapatkan persetujuan untuk mengikuti seminar proposal dari dosen pembimbing melalui modul ini. Mahasiswa harus mengirimkan file submission berupa:
 * File Proposal dengan format: PROPOSAL_NIM.pdf
 * File Form Persetujuan Seminar Proposal yang sudah diisi dan ditandatangani oleh Dosen Pembimbing dengan format: DOPING_NIM.pdf beserta dengan lembar kendali bimbingan pra proposal yang telah ditandatangani oleh dosen pembimbing

2. **Workshop Seminar Proposal**: Setelah persetujuan seminar proposal diterima, mahasiswa harus melakukan submission kembali dengan proposal yang sudah siap dengan format: Proposal_NIM.pdf. Phase submission ditutup selama seminar proposal berlangsung dan akan dibuka kembali setelah seminar proposal selesai.

3. **Persetujuan Seminar Hasil**: Setelah proposal disetujui, mahasiswa dapat melanjutkan ke seminar hasil. Mahasiswa harus mengirimkan file submission berupa:
 * File Persetujuan Seminar Hasil yang telah diisi dan ditandatangani oleh Dosen Pembimbing 1 dan 2 dengan format: PERSETUJUAN_NIM.pdf
 * Form lembar kendali bimbingan pra seminar hasil yang telah ditandatangani oleh Dosen Pembimbing 1 dan 2

4. **Seminar Hasil**: Setelah persetujuan seminar hasil diterima, mahasiswa dapat melanjutkan ke seminar hasil. Mahasiswa harus mengirimkan file submission berupa:
 * File Proposal dengan format: SKRIPSI_NIM.pdf
 * File Berita Acara Seminar Hasil dengan format: BASemHas_NIM.doc/docx yang telah terisi kecuali pada bagian nilai

5. **Sidang Meja Hijau**: Setelah seminar hasil selesai, mahasiswa dapat melanjutkan ke sidang meja hijau. Mahasiswa harus mengirimkan file submission berupa:
 * File Proposal dengan format: SKRIPSI_NIM.pdf
 * File Persetujuan Sidang Meja Hijau yang telah diisi Nama Dosen Pembimbing 1 dan 2, Nama Dosen Pembanding 1 dan 2 dengan format: PERSETUJUAN_NIM.pdf dan wajib ditandatangani oleh dosen pembimbing dan dosen Pembanding
 * Form lembar kendali bimbingan pra sidang meja hijau yang telah ditandatangani oleh Dosen Pembimbing 1 dan 2

Dengan mengikuti langkah-langkah di atas, mahasiswa dapat maju ke seminar proposal dan melanjutkan ke seminar hasil dan sidang meja hijau.


⏳ Memproses ...

🔍 PERTANYAAN:
apa langkah-langkah maju seminar hasil?

🤖 JAWABAN  (meta/llama-3.1-8b-instruct):


Berikut adalah langkah-langkah maju seminar hasil berdasarkan pedoman yang disediakan:

1. **Tahap Bimbingan dan Penulisan Skripsi**:
 * Mulai: Proses diawali oleh Mahasiswa.
 * Bimbingan Awal: Mahasiswa melakukan Bimbingan Tugas Akhir dengan Dosen Pembimbing 1 dan 2.
 * Penulisan: Mahasiswa melanjutkan ke tahap Penulisan Skripsi.
 * Pengujian: Mahasiswa melakukan Uji Program dengan Dosen Pembimbing.

2. **Tahap Cek Plagiat dan Validasi Dosen Pembimbing**:
 * Pengajuan: Mahasiswa melakukan Pengajuan Seminar Hasil.
 * Verifikasi Plagiat: Pihak Prodi / Fakultas melakukan Cek Plagiat.
 * Percabangan (Similarity ≤ 25%?):
  * Jika TIDAK: Mahasiswa harus kembali ke tahap Penulisan Skripsi.
  * Jika YA: Dosen Pembimbing (DOPING) akan memberikan Acc Seminar Hasil.

3. **Tahap Pendaftaran dan Pelaksanaan Seminar Hasil**:
 * Penyusunan Berkas: Berkas masuk ke ranah Prodi / Fakultas sebagai Berkas Seminar Hasil.
 * Percabangan (Berkas Lengkap?):
  * Jika TIDAK: Mahasiswa harus kembali ke tahap Pengajuan Seminar Hasil.
  * Jika YA: Proses berlanjut ke pelaksanaan Seminar Hasil oleh Prodi / Fakultas.
 * Percabangan (Lulus Seminar Hasil?):
  * Jika TIDAK: Mahasiswa harus melakukan Perbaikan Skripsi dan kembali ke tahap Pengajuan Seminar Hasil.
  * Jika YA: Mahasiswa dapat melanjutkan ke tahap berikutnya.

4. **Tahap Sidang Sarjana (Meja Hijau) hingga Selesai**:
 * Persiapan: Mahasiswa melakukan Persiapan Sidang Sarjana (setelah melalui tahap Perbaikan Skripsi jika ada catatan dari seminar).
 * Pelaksanaan Sidang: Pihak Prodi / Fakultas menyelenggarakan Sidang Sarjana.
 * Percabangan (Lulus Sidang Sarjana?):
  * Jika TIDAK: Mahasiswa harus kembali ke tahap Persiapan Sidang Sarjana.
  * Jika YA: Pihak Prodi / Fakultas melakukan Finalisasi Skripsi.

Dengan demikian, langkah-langkah maju seminar hasil dapat dilakukan dengan mengikuti tahapan-tahapan yang telah dijelaskan di atas.

👋 Sesi selesai.
